In [1]:
import sys
sys.path.append('../..')

import torch
import numpy as np
import pickle
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from repe import repe_pipeline_registry

# Register RepE pipelines
repe_pipeline_registry()

# Import ACT modules
from examples.act_new.act_core import EPA
from examples.act_new.epa_calibration import (
    BehaviorPromptGenerator,
    LinearRegressionCalibrator,
    AffineCalibrator,
    CalibrationCoefficients,
    CONVERSATIONAL_BEHAVIORS,
)
from examples.act_new.utils import (
    read_epa_scores,
    format_for_reading,
)

print("Modules loaded successfully!")

c:\Users\Kyra\mambaforge-pypy3\envs\repeng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modules loaded successfully!


In [2]:
# Load the extracted EPA directions
directions_path = "epa_directions.pkl"

with open(directions_path, 'rb') as f:
    directions_data = pickle.load(f)

rep_readers = directions_data['rep_readers']
hidden_layers = directions_data['hidden_layers']
model_name = directions_data['model_name']

print(f"Loaded directions for model: {model_name}")
print(f"Dimensions available: {list(rep_readers.keys())}")
print(f"Number of layers: {len(hidden_layers)}")

Loaded directions for model: meta-llama/Llama-3.1-8B-Instruct
Dimensions available: ['evaluation', 'potency', 'activity']
Number of layers: 31


In [3]:
# Load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {model_name}")

Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.10s/it]


Model loaded: meta-llama/Llama-3.1-8B-Instruct


In [4]:
# Create rep-reading pipeline
rep_pipeline = pipeline("rep-reading", model=model, tokenizer=tokenizer)
print("Rep-reading pipeline created!")

Device set to use cuda:0


Rep-reading pipeline created!


In [5]:
from examples.act_new.epa_calibration import BehaviorPromptGenerator
from examples.act_new.act_core import EPA
from utils import format_llama3_prompt
generator = BehaviorPromptGenerator()
# Define your LLM generation function
def epa_to_description(epa: EPA) -> str:
    """Convert EPA values to descriptive terms."""
    
    def scale_term(value: float, neg_terms: list, pos_terms: list) -> str:
        # EPA typically ranges from about -4.3 to +4.3
        # Map to: very [neg], [neg], somewhat [neg], neither, somewhat [pos], [pos], very [pos]
        if value <= -3.0:
            return neg_terms[0]      # "very bad"
        elif value <= -1.5:
            return neg_terms[1]      # "bad"  
        elif value <= -0.5:
            return neg_terms[2]      # "somewhat bad"
        elif value <= 0.5:
            return neg_terms[3]      # "neither good nor bad"
        elif value <= 1.5:
            return pos_terms[0]      # "somewhat good"
        elif value <= 3.0:
            return pos_terms[1]      # "good"
        else:
            return pos_terms[2]      # "very good"
    
    e_desc = scale_term(epa.e, 
        ["very bad", "bad", "somewhat bad", "neither good nor bad"],
        ["somewhat good", "good", "very good"])
    p_desc = scale_term(epa.p,
        ["very impotent", "impotent", "somewhat impotent", "neither potent nor impotent"],
        ["somewhat potent", "potent", "very potent"])
    a_desc = scale_term(epa.a,
        ["very passive", "passive", "somewhat passive", "neither active nor passive"],
        ["somewhat active", "active", "very active"])
    
    return f"{e_desc}, {p_desc}, and {a_desc}"

def generate_with_llm(behavior: str, epa: EPA) -> str:
    epa_description = epa_to_description(epa)

    behavior = behavior.replace("_", " ")
    
    prompt = format_llama3_prompt(
        "You are simulating a human person engaging in a conversation with another human person.",
        f"""Generate a single conversational utterance that embodies the behavior '{behavior}' with an affective tone that is {epa_description}. Respond with only the utterance, no explanation or quotation marks."""
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Strip quotes and whitespace
    response = response.strip().strip('"\'')
    
    return response
# Set it on the generator
generator.set_llm_generate_function(generate_with_llm)
# Now generate_utterance will use the LLM for behaviors without templates
utterance = generator.generate_utterance("accuse")  # Will call LLM
print(utterance)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


You're always sabotaging my plans and blaming it on me when it falls apart.


In [6]:
# Initialize the behavior prompt generator
# generator = BehaviorPromptGenerator()

print(f"Total behaviors in dictionary: {len(generator.behaviors)}")
print(f"Conversational behaviors available: {len(generator.available_behaviors)}")
print(f"\nSample behaviors: {generator.available_behaviors[:10]}")

Total behaviors in dictionary: 853
Conversational behaviors available: 85

Sample behaviors: ['accuse', 'admire', 'advise', 'agree_with', 'alarm', 'amaze', 'apologize_to', 'appeal_to', 'appreciate', 'argue_with']


In [7]:
# Show some behavior EPA values from the dictionary
print("Sample behavior EPA values from dictionary:")
print("-" * 50)
for behavior in ['thank', 'beg', 'threaten', 'comfort', 'criticize']:
    epa = generator.get_behavior_epa(behavior)
    if epa:
        print(f"{behavior:15} E={epa.e:+.2f} P={epa.p:+.2f} A={epa.a:+.2f}")

Sample behavior EPA values from dictionary:
--------------------------------------------------
thank           E=+3.18 P=+2.17 A=+0.78
beg             E=-1.42 P=-2.22 A=+0.81
threaten        E=-2.88 P=+0.73 A=+1.83
comfort         E=+3.11 P=+2.08 A=-0.72
criticize       E=-1.65 P=+0.07 A=+1.24


In [8]:
# Generate calibration pairs (utterance, target_EPA)
n_calibration_samples = 85  # Adjust based on available time/compute

calibration_pairs = generator.generate_calibration_pairs(
    n_samples=n_calibration_samples,
    behaviors=generator.available_behaviors[:20]  # Use first 20 behaviors with templates
) ### CALIBRATION PAIRS NEED TO ACTUALLY BE GENERATED

print(f"Generated {len(calibration_pairs)} calibration pairs")
print("\nSample pairs:")
for utterance, target_epa in calibration_pairs[:3]:
    print(f"  '{utterance[:600]}...'")
    print(f"    Target: {target_epa}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Generated 85 calibration pairs

Sample pairs:
  'You're just covering this up because you're in on it with them aren't you?...'
    Target: EPA(e=-1.35, p=1.05, a=1.39)
  'She's an incredibly talented artist, don't you think her use of color is just breathtaking?...'
    Target: EPA(e=1.99, p=0.67, a=-0.26)
  'You're doing a great job on this project, keep pushing forward and don't hesitate to ask for help when you need it....'
    Target: EPA(e=2.10, p=1.61, a=0.30)
